# Improved SPY Direction Prediction from S&P 500 Breadth

This notebook is a second experiment kept separate from the original project notebook.

Goal: next-day `SPY` direction prediction by adding richer breadth features, lagged predictors, and a chronological validation step before the final test.

## Plan

1. Load the raw Yahoo Finance CSVs
2. Build an enhanced feature set
3. Split the data into train, validation, and test periods
4. Measure a simple baseline
5. Fit a linear regression model
6. Fit tree-based models
7. Compare the results

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 140)

possible_data_dirs = [
    Path("data"),
    Path("../data"),
    Path.cwd() / "data",
    Path.cwd().parent / "data",
]

DATA_DIR = None
for candidate in possible_data_dirs:
    if (candidate / "sp500_close.csv").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find the data folder containing sp500_close.csv")

CLOSE_PATH = DATA_DIR / "sp500_close.csv"
VOLUME_PATH = DATA_DIR / "sp500_volume.csv"

print("Using data directory:", DATA_DIR.resolve())

Using data directory: /Users/vrundpatel/2600-Final/2600/data


In [5]:
close = pd.read_csv(CLOSE_PATH, index_col=0, parse_dates=True)
volume = pd.read_csv(VOLUME_PATH, index_col=0, parse_dates=True)

print("close shape :", close.shape)
print("volume shape:", volume.shape)
print("first date  :", close.index.min().date())
print("last date   :", close.index.max().date())
print("SPY present :", "SPY" in close.columns)

close shape : (2089, 504)
volume shape: (2089, 504)
first date  : 2018-01-02
last date   : 2026-04-24
SPY present : True


## Enhanced Predictors

The original notebook used a small set of breadth features.

This improved experiment adds:

- extra moving-average breadth signals like `pct_above_50dma`
- more `SPY` momentum and volatility features
- rolling averages of key breadth measures
- lagged versions of important predictors

In [6]:
def build_enhanced_dataset(close_df, volume_df):
    spy_close = close_df["SPY"].copy()
    stock_close = close_df.drop(columns=["SPY"]).copy()
    stock_volume = volume_df[stock_close.columns].copy()

    min_history = int(len(stock_close) * 0.80)
    keep_mask = stock_close.notna().sum() >= min_history

    stock_close = stock_close.loc[:, keep_mask].copy().ffill()
    stock_volume = stock_volume.loc[:, stock_close.columns].copy().ffill()

    stock_returns = stock_close.pct_change(fill_method=None)
    spy_returns = spy_close.pct_change(fill_method=None)
    volume_change = stock_volume.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)

    stock_ma_5 = stock_close.rolling(5).mean()
    stock_ma_20 = stock_close.rolling(20).mean()
    stock_ma_50 = stock_close.rolling(50).mean()

    dataset = pd.DataFrame(index=close_df.index)
    dataset["breadth_up_pct"] = stock_returns.gt(0).mean(axis=1)
    dataset["breadth_down_pct"] = stock_returns.lt(0).mean(axis=1)
    dataset["avg_stock_return"] = stock_returns.mean(axis=1)
    dataset["median_stock_return"] = stock_returns.median(axis=1)
    dataset["return_dispersion"] = stock_returns.std(axis=1)
    dataset["avg_volume_change"] = volume_change.mean(axis=1)
    dataset["pct_above_5dma"] = stock_close.gt(stock_ma_5).where(stock_ma_5.notna()).mean(axis=1)
    dataset["pct_above_20dma"] = stock_close.gt(stock_ma_20).where(stock_ma_20.notna()).mean(axis=1)
    dataset["pct_above_50dma"] = stock_close.gt(stock_ma_50).where(stock_ma_50.notna()).mean(axis=1)
    dataset["spy_return_today"] = spy_returns
    dataset["spy_return_3d"] = spy_close.pct_change(3)
    dataset["spy_return_5d"] = spy_close.pct_change(5)
    dataset["spy_return_10d"] = spy_close.pct_change(10)
    dataset["spy_vol_5d"] = spy_returns.rolling(5).std()
    dataset["spy_vol_20d"] = spy_returns.rolling(20).std()
    dataset["breadth_spread"] = dataset["breadth_up_pct"] - dataset["breadth_down_pct"]
    dataset["breadth_momentum_3d"] = dataset["breadth_up_pct"].diff(3)
    dataset["breadth_momentum_5d"] = dataset["breadth_up_pct"].diff(5)
    dataset["avg_return_5d_mean"] = dataset["avg_stock_return"].rolling(5).mean()
    dataset["dispersion_5d_mean"] = dataset["return_dispersion"].rolling(5).mean()
    dataset["volume_change_5d_mean"] = dataset["avg_volume_change"].rolling(5).mean()

    lag_sources = [
        "breadth_up_pct",
        "breadth_down_pct",
        "avg_stock_return",
        "median_stock_return",
        "return_dispersion",
        "avg_volume_change",
        "pct_above_5dma",
        "pct_above_20dma",
        "pct_above_50dma",
        "spy_return_today",
        "spy_return_3d",
        "spy_return_5d",
        "spy_return_10d",
        "spy_vol_5d",
        "spy_vol_20d",
        "breadth_spread",
    ]

    for column in lag_sources:
        for lag in [1, 2, 3]:
            dataset[f"{column}_lag{lag}"] = dataset[column].shift(lag)

    dataset["next_spy_return"] = spy_returns.shift(-1)
    dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()
    dataset["target_up_tomorrow"] = (dataset["next_spy_return"] > 0).astype(int)
    dataset = dataset.drop(columns=["next_spy_return"])

    feature_columns = [column for column in dataset.columns if column != "target_up_tomorrow"]
    return dataset, feature_columns

In [7]:
model_df, feature_cols = build_enhanced_dataset(close, volume)

print("model rows    :", len(model_df))
print("feature count :", len(feature_cols))
print("first model date:", model_df.index.min().date())
print("last model date :", model_df.index.max().date())

model_df.iloc[:5, :10]

model rows    : 2036
feature count : 69
first model date: 2018-03-19
last model date : 2026-04-23


,breadth_up_pct,breadth_down_pct,avg_stock_return,median_stock_return,return_dispersion,avg_volume_change,pct_above_5dma,pct_above_20dma,pct_above_50dma,spy_return_today
Date,,,,,,,,,,
2018-03-19,0.103093,0.874227,-0.010998,-0.009738,0.010428,-0.303215,0.232704,0.538784,0.51153,-0.013531
2018-03-20,0.593814,0.373196,0.002079,0.002492,0.012621,-0.056200,0.404612,0.572327,0.526205,0.001701
2018-03-21,0.414433,0.562887,-0.000295,-0.001491,0.012949,0.085594,0.398323,0.551363,0.501048,-0.001920
2018-03-22,0.090722,0.890722,-0.022290,-0.022846,0.015998,0.327863,0.073375,0.245283,0.318658,-0.024997
2018-03-23,0.065979,0.915464,-0.019098,-0.019170,0.013666,0.452215,0.044025,0.111111,0.209644,-0.021315


In [8]:
total_rows = len(model_df)
trainval_end = int(total_rows * 0.80)
trainval_df = model_df.iloc[:trainval_end].copy()
test_df = model_df.iloc[trainval_end:].copy()

validation_start = int(len(trainval_df) * 0.80)
train_df = trainval_df.iloc[:validation_start].copy()
validation_df = trainval_df.iloc[validation_start:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_up_tomorrow"]
X_validation = validation_df[feature_cols]
y_validation = validation_df["target_up_tomorrow"]
X_trainval = trainval_df[feature_cols]
y_trainval = trainval_df["target_up_tomorrow"]
X_test = test_df[feature_cols]
y_test = test_df["target_up_tomorrow"]

print("train rows     :", len(train_df))
print("validation rows:", len(validation_df))
print("test rows      :", len(test_df))
print()
print("train range     :", train_df.index.min().date(), "to", train_df.index.max().date())
print("validation range:", validation_df.index.min().date(), "to", validation_df.index.max().date())
print("test range      :", test_df.index.min().date(), "to", test_df.index.max().date())

train rows     : 1302
validation rows: 326
test rows      : 408

train range     : 2018-03-19 to 2023-05-18
validation range: 2023-05-19 to 2024-09-05
test range      : 2024-09-06 to 2026-04-23


## Baseline

The baseline predicts the majority class from the training data every time.

If a more complex model cannot beat this baseline on the held-out test period, then the extra modeling work is not helping.

In [9]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

majority_class = int(y_train.mode()[0])
baseline_test_pred = np.full(len(y_test), majority_class)
baseline_test_metrics = evaluate_predictions(y_test, baseline_test_pred)
baseline_test_cm = confusion_matrix(y_test, baseline_test_pred)

print("Baseline majority class:", majority_class)
baseline_test_metrics

Baseline majority class: 1


{'accuracy': 0.571078431372549,
 'precision': 0.571078431372549,
 'recall': 1.0,
 'f1': 0.7269890795631825}

In [10]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LinearRegression


In [11]:
def tune_threshold(y_true, scores, n_grid=201):
    """Pick a threshold on validation: maximize balanced accuracy, then F1.

    Optimizing plain accuracy leans on the more common class (often 'up') and
    often picks a very low threshold, which drives recall near 100%. Balanced
    accuracy (average of sensitivity and specificity) discourages a trivial
    all-positive rule.
    """
    scores = np.asarray(scores, dtype=float)
    lo, hi = float(scores.min()), float(scores.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        pred = (scores >= 0.5).astype(int)
        return 0.5, evaluate_predictions(y_true, pred)
    best_threshold = 0.5 * (lo + hi)
    best_metrics = None
    best_key = None
    for t in np.linspace(lo, hi, n_grid):
        pred = (scores >= t).astype(int)
        m = evaluate_predictions(y_true, pred)
        key = (m["balanced_accuracy"], m["f1"])
        if best_key is None or key > best_key:
            best_key = key
            best_threshold = float(t)
            best_metrics = m
    return best_threshold, best_metrics

## Linear Regression Model

Because the target is `0` or `1`, linear regression is being used here as a **linear probability model**.

It predicts a continuous score, and then we turn that score into a class prediction. The **validation** threshold is chosen to maximize **balanced accuracy** (then F1 as a tie-break), not raw accuracy, so the cutoff is less biased toward always predicting the majority class.

In [12]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_validation_scores = linear_model.predict(X_validation)
linear_threshold, linear_validation_metrics = tune_threshold(y_validation, linear_validation_scores)

linear_model.fit(X_trainval, y_trainval)
linear_test_scores = linear_model.predict(X_test)
linear_test_pred = (linear_test_scores >= linear_threshold).astype(int)
linear_test_metrics = evaluate_predictions(y_test, linear_test_pred)
linear_test_cm = confusion_matrix(y_test, linear_test_pred)

print("Linear regression threshold:", round(linear_threshold, 2))
print("Validation metrics:")
print(linear_validation_metrics)
print()
print("Test metrics:")
linear_test_metrics

Linear regression threshold: 0.25
Validation metrics:
{'accuracy': 0.5766871165644172, 'precision': 0.5766871165644172, 'recall': 1.0, 'f1': 0.7315175097276264}

Test metrics:


{'accuracy': 0.5808823529411765,
 'precision': 0.5767326732673267,
 'recall': 1.0,
 'f1': 0.7315541601255887}

## Tree-Based Models

Tree-based models can capture nonlinear relationships and feature interactions more easily than a simple linear regression model.

Here we compare three tree-style approaches:

- `RandomForestClassifier`
- `ExtraTreesClassifier`
- `GradientBoostingClassifier`

In [13]:
tree_models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=700,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
        n_estimators=250,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.8,
    ),
}

tree_validation_rows = []
tree_threshold_lookup = {}

for model_name, model in tree_models.items():
    model.fit(X_train, y_train)
    validation_scores = model.predict_proba(X_validation)[:, 1]
    best_threshold, best_metrics = tune_threshold(y_validation, validation_scores)

    tree_threshold_lookup[model_name] = best_threshold
    tree_validation_rows.append({
        "model": model_name,
        "threshold": best_threshold,
        "val_accuracy": best_metrics["accuracy"],
        "val_balanced_accuracy": best_metrics["balanced_accuracy"],
        "val_precision": best_metrics["precision"],
        "val_recall": best_metrics["recall"],
        "val_f1": best_metrics["f1"],
    })

tree_validation_results_df = pd.DataFrame(tree_validation_rows).sort_values(
    ["val_balanced_accuracy", "val_f1"], ascending=False
)
tree_validation_results_df

,model,threshold,val_accuracy,val_precision,val_recall,val_f1
1,ExtraTrees,0.48,0.582822,0.580745,0.994681,0.733333
2,GradientBoosting,0.36,0.582822,0.581250,0.989362,0.732283
0,RandomForest,0.45,0.579755,0.578462,1.000000,0.732943


In [14]:
best_tree_name = tree_validation_results_df.iloc[0]["model"]
best_tree_threshold = tree_threshold_lookup[best_tree_name]
best_tree_model = tree_models[best_tree_name]

best_tree_model.fit(X_trainval, y_trainval)
best_tree_test_scores = best_tree_model.predict_proba(X_test)[:, 1]
best_tree_test_pred = (best_tree_test_scores >= best_tree_threshold).astype(int)
best_tree_test_metrics = evaluate_predictions(y_test, best_tree_test_pred)
best_tree_test_cm = confusion_matrix(y_test, best_tree_test_pred)

print("Best tree model:", best_tree_name)
print("Tree threshold:", round(best_tree_threshold, 2))
print("Tree test metrics:")
best_tree_test_metrics

Best tree model: ExtraTrees
Tree threshold: 0.48
Tree test metrics:


{'accuracy': 0.5784313725490197,
 'precision': 0.5768261964735516,
 'recall': 0.9828326180257511,
 'f1': 0.726984126984127}

In [15]:
comparison_df = pd.DataFrame(
    [baseline_test_metrics, linear_test_metrics, best_tree_test_metrics],
    index=["Baseline", "Linear Regression", best_tree_name],
)

comparison_df

,accuracy,precision,recall,f1
Baseline,0.571078,0.571078,1.000000,0.726989
Linear Regression,0.580882,0.576733,1.000000,0.731554
ExtraTrees,0.578431,0.576826,0.982833,0.726984


In [16]:
print("Baseline confusion matrix:\n", baseline_test_cm)
print()
print("Linear regression confusion matrix:\n", linear_test_cm)
print()
print(f"{best_tree_name} confusion matrix:\n", best_tree_test_cm)

Baseline confusion matrix:
 [[  0 175]
 [  0 233]]

Linear regression confusion matrix:
 [[  4 171]
 [  0 233]]

ExtraTrees confusion matrix:
 [[  7 168]
 [  4 229]]


In [17]:
top_tree_signals = pd.Series(best_tree_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(12)
top_tree_signals

breadth_down_pct_lag3    0.020362
breadth_spread_lag3      0.020079
breadth_up_pct_lag1      0.019723
breadth_up_pct_lag3      0.019519
pct_above_5dma_lag2      0.019356
breadth_momentum_3d      0.019283
breadth_down_pct_lag1    0.019219
breadth_spread_lag1      0.019009
pct_above_5dma_lag1      0.018425
breadth_down_pct_lag2    0.018195
pct_above_5dma_lag3      0.018172
breadth_up_pct           0.017889
dtype: float64

## Conclusion

This improved notebook shows both a **linear regression model** and **tree-based models** on the same richer predictor set.

The final comparison should answer three questions:

1. Did linear regression beat the baseline?
2. Did the best tree-based model beat the baseline?
3. Which one performed better on the held-out test period?

That gives a cleaner class presentation because it compares a simple linear model against a more flexible tree-style approach.